<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 5장 보너스 코드

## PyTorch state dict로부터의 대체 가중치 로딩

- 메인 챕터에서는 OpenAI로부터 직접 GPT 모델 가중치를 로드했습니다
- 이 노트북은 원본 TensorFlow 파일로부터 제가 생성하여 [Hugging Face Model Hub](https://huggingface.co/docs/hub/en/models-the-hub)의 [https://huggingface.co/rasbt/gpt2-from-scratch-pytorch](https://huggingface.co/rasbt/gpt2-from-scratch-pytorch)에 업로드한 PyTorch state dict 파일로부터 모델 가중치를 로드하는 대체 가중치 로딩 코드를 제공합니다
- 이는 5장에서 설명한 state-dict 방법을 통해 PyTorch 모델의 가중치를 로드하는 것과 개념적으로 동일합니다:

```python
state_dict = torch.load("model_state_dict.pth")
model.load_state_dict(state_dict) 
```

### 모델 선택

In [ ]:
from importlib.metadata import version

pkgs = ["torch"]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
BASE_CONFIG = {
    "vocab_size": 50257,    # 어휘 크기(vocabulary size)
    "context_length": 1024, # 컨텍스트 길이(context length)
    "drop_rate": 0.0,       # 드롭아웃 비율(dropout rate)
    "qkv_bias": True        # 쿼리-키-값 편향(query-key-value bias)
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}


CHOOSE_MODEL = "gpt2-small (124M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

### 파일 다운로드

In [ ]:
file_name = "gpt2-small-124M.pth"
# file_name = "gpt2-medium-355M.pth"
# file_name = "gpt2-large-774M.pth"
# file_name = "gpt2-xl-1558M.pth"

In [ ]:
import os
import urllib.request

url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

if not os.path.exists(file_name):
    urllib.request.urlretrieve(url, file_name)
    print(f"Downloaded to {file_name}")

### 가중치 로드

In [ ]:
import torch
from llms_from_scratch.ch04 import GPTModel
# llms_from_scratch 설치 지침은 다음을 참조하세요:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg


gpt = GPTModel(BASE_CONFIG)
gpt.load_state_dict(torch.load(file_name, weights_only=True))
gpt.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt.to(device);

### 텍스트 생성

In [ ]:
import tiktoken
from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text


torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate(
    model=gpt.to(device),
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    top_k=1,
    temperature=1.0
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

## 대체 safetensors 파일

- 또한, [https://huggingface.co/rasbt/gpt2-from-scratch-pytorch](https://huggingface.co/rasbt/gpt2-from-scratch-pytorch) 저장소에는 소위 `.safetensors` 버전의 state dict가 포함되어 있습니다
- `.safetensors` 파일의 매력은 보안적 설계에 있으며, 텐서 데이터만 저장하고 로딩 중 잠재적으로 악의적인 코드의 실행을 방지합니다
- 최신 버전의 PyTorch(예: 2.0 이상)에서는 `torch.load`와 함께 `weights_only=True` 인수를 사용하여 (예: `torch.load("model_state_dict.pth", weights_only=True)`) 코드 실행을 건너뛰고 가중치만 로드하여 보안을 향상시킬 수 있습니다 (이는 현재 PyTorch 2.6 이상에서 기본값으로 활성화됨); 그러므로 그 경우 state dict 파일로부터 가중치를 로드하는 것은 더 이상 우려사항이 아닙니다
- 그러나 아래 코드 블록은 이러한 `.safetensor` 파일로부터 모델을 로드하는 방법을 간략하게 보여줍니다

In [ ]:
file_name = "gpt2-small-124M.safetensors"
# file_name = "gpt2-medium-355M.safetensors"
# file_name = "gpt2-large-774M.safetensors"
# file_name = "gpt2-xl-1558M.safetensors"

In [ ]:
import os
import urllib.request

url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

if not os.path.exists(file_name):
    urllib.request.urlretrieve(url, file_name)
    print(f"Downloaded to {file_name}")

In [ ]:
# 파일 로드

from safetensors.torch import load_file

gpt = GPTModel(BASE_CONFIG)
gpt.load_state_dict(load_file(file_name))
gpt.eval();

In [ ]:
token_ids = generate(
    model=gpt.to(device),
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    top_k=1,
    temperature=1.0
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))